# Chapter 8 — Similarity, Recommendation, Clustering
**MADT6004 · Brew Lab BKK case**

Two related ideas in unsupervised learning:

- **Clustering** — group similar customers so you can talk about "types"
- **Recommendation** — measure similarity between *items* and recommend products that go together

You will:
1. Build a customer single-view (RFM-like features)
2. Run k-means and pick k by silhouette
3. Compute item-item cosine similarity from co-purchase patterns


## 0. Bootstrap (Colab + local)

In [ ]:
# Bootstrap — make sure brewlab.db is available, both locally and in Colab.
import os
DB_CANDIDATES = [
    "../../Integrated Data Analytics Exercise/data/brewlab.db",
    "MADT6004/Integrated Data Analytics Exercise/data/brewlab.db",
]
DB_PATH = next((p for p in DB_CANDIDATES if os.path.exists(p)), None)
if DB_PATH is None:
    if not os.path.exists("MADT6004"):
        os.system("git clone -q https://github.com/thanachart/MADT6004.git")
    os.system("pip install -q -r 'MADT6004/Integrated Data Analytics Exercise/requirements.txt'")
    DB_PATH = "MADT6004/Integrated Data Analytics Exercise/data/brewlab.db"
print("DB:", DB_PATH)


## 1. Setup

In [ ]:
import sqlite3
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity

conn = sqlite3.connect(DB_PATH)
print("Tables:", [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()])


## 2. Customer single view

In [ ]:
sv = pd.read_sql("""
SELECT c.customer_id,
       COUNT(t.order_id) AS visits,
       COALESCE(SUM(t.total), 0) AS spend,
       COALESCE(AVG(t.total), 0) AS avg_ticket,
       MAX(date(t.datetime))      AS last_visit
FROM customers c LEFT JOIN transactions t ON c.customer_id = t.customer_id
GROUP BY c.customer_id
""", conn)
sv["last_visit"] = pd.to_datetime(sv["last_visit"])
ref = sv["last_visit"].max()
sv["recency_days"] = (ref - sv["last_visit"]).dt.days.fillna(9999).astype(int)
sv = sv[sv["visits"] > 0]   # active customers only
print(sv.head())
print("Active customers:", len(sv))


## 3. k-means + silhouette
Standardize the features, then sweep k from 2 to 7.

In [ ]:
feats = ["visits", "spend", "avg_ticket", "recency_days"]
Xs = StandardScaler().fit_transform(sv[feats])

scores = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xs)
    scores.append((k, silhouette_score(Xs, km.labels_)))

print("k    silhouette")
for k, s in scores: print(f"{k}    {s:.3f}")

best_k = max(scores, key=lambda x: x[1])[0]
print(f"\nBest k by silhouette: {best_k}")

km = KMeans(n_clusters=best_k, n_init=10, random_state=42).fit(Xs)
sv["cluster"] = km.labels_


## 4. Cluster profiles

In [ ]:
profile = sv.groupby("cluster")[feats].mean().round(1)
profile["n"] = sv.groupby("cluster").size()
print(profile)


## 5. Item-item similarity
Build a customer × product purchase matrix, then compute cosine similarity between products to power a "Customers who bought X also bought Y" recommender.

In [ ]:
co = pd.read_sql("""
SELECT t.customer_id, oi.product_id, SUM(oi.qty) AS qty
FROM order_items oi JOIN transactions t ON oi.order_id = t.order_id
GROUP BY t.customer_id, oi.product_id
""", conn)
mat = co.pivot_table(index="customer_id", columns="product_id", values="qty", fill_value=0)

prods = pd.read_sql("SELECT product_id, name FROM products", conn).set_index("product_id")["name"]
sim = pd.DataFrame(cosine_similarity(mat.T), index=mat.columns, columns=mat.columns)

def top_similar(pid, n=5):
    s = sim.loc[pid].drop(pid).sort_values(ascending=False).head(n)
    return pd.DataFrame({"product": prods.loc[s.index].values, "similarity": s.values.round(3)})

# Pick a popular product to demo
example_pid = co.groupby("product_id")["qty"].sum().idxmax()
print("Example product:", prods.loc[example_pid])
print("\nTop 5 most similar:")
print(top_similar(example_pid))


## Discussion prompts
1. The silhouette score isn't always pretty. What does a low silhouette tell you about how separable the clusters really are?
2. Cluster names matter. Look at the profiles and write a short, plain-English label for each cluster.
3. Item-item recommendations are simple but powerful. Where would they break down?
